In [1]:
from langchain_community.document_loaders import DirectoryLoader
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context, conditional
from langchain_ollama import OllamaEmbeddings, ChatOllama


In [2]:
data = 'data'
language = "Português"
distributions = {
    simple:0.4,
    reasoning:0.2,
    multi_context:0.2,
    conditional:0.2
    }


In [3]:
loader = DirectoryLoader(data)
documents = loader.load()

for document in documents:
    document.metadata['filename'] = document.metadata['source']

In [4]:
model = ChatOllama(model='llama3.1')
embeddings = OllamaEmbeddings(model='llama3.1')

generator = TestsetGenerator.from_langchain(
    model,
    model,
    embeddings
)


generator.adapt(language, evolutions=[simple, reasoning, conditional,multi_context])
generator.save(evolutions=[simple, reasoning, multi_context, conditional])


In [5]:
testset = generator.generate_with_langchain_docs(documents, 128, distributions, with_debugging_logs=True)

embedding nodes:   0%|          | 0/4 [00:00<?, ?it/s]

Generating:   0%|          | 0/2 [00:00<?, ?it/s]

[ragas.testset.filters.DEBUG] context scoring: {'clarity': 2, 'depth': 3, 'structure': 3, 'relevance': 1, 'score': 2.25}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['Digital Signal Processor', 'Sustentabilidade Auditada', 'Regulação Estática']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 2, 'depth': 3, 'structure': 1, 'relevance': 2, 'score': 2.0}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['Digital Signal Processor', 'Sustentabilidade Auditada', 'Regulação Estática']
[ragas.testset.evolutions.INFO] seed question generated: Here's the question based on the given context:

"São quais as características do sistema de sustentabilidade auditado presente na composição?"
[ragas.testset.evolutions.INFO] seed question generated: Here is the question that can be fully answered from the given context:

"Qual é a precisão de regulação estática da saída do dispositivo?" 

The answer to this question can be found in the section "Saída" of the provided

In [6]:
dataframe = testset.to_pandas()
dataframe.to_csv('testset.csv')

In [7]:
dataset = testset.to_dataset()
dataset.save_to_disk('testset')

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]